In [ ]:
"""
Token Embeddings in the simplest form 
it represents the individual token in the dimensions you provide so it acts like a look up table of ,in how many ways the token can be identified
the parameters are the 
vocab_size -> this represents how many rows (as the total size of the dictionary where the model can use this generating the words)
num_dims -> this is the number of dimensions in which the individual word is represented 


def parameters(self):
        return [self.weights]

we need to explicitly return the params ,or else optimizer can't be able to track any parameters to change in the backprop

actually custom backward is not necessary if you inherited the nn.Module
if you wanted to take a look about how do they change and then you can give a try for them

the indices shap -> (batch_size)
so the indices shape is the batch_size right and then 
 what was left is that
now we need to think of the grad_out shape right 
we get the dimensions out from the indices
so the grad_out shape is the (batch_size,num_dims)

weights.grad shape -> (vocab,num_dims)

out = weight[indices]

output CHANGES based on WHICH indices are used
When we differentiate, we're looking at the RELATIONSHIP between output and weights
The indices act as a "selector" - they determine WHICH weights affect the output

If you change indices → you get DIFFERENT rows from weights (discrete change - can't differentiate!)
If you change weights → the values in those rows change (continuous change - CAN differentiate!)

actually what is the learnable one?weights right not the indices 
so we do use the backprop on the wieghts not the weights this is the simple way to understand


loss/weights = loss/out * out/weights (we are using the chain rule here)
loss/out = grad_out


self.weights.grad -> so this is what needed to be change right and then we can change this by the
so bro we did actually need to know is that
okay,we know weights needed to be changed right
0        _ _ _ _ .......... _
1        _ _ _ _ .......... _
2        _ _ _ _ .......... _
3        _ _ _ _ .......... _
so in this way the weights actually are
and then when we bring back the losses and then we will bring back with the indices right
so that we can learn to see which indices have actually needed to learn
so we got back the two indices 0,2 only and then we need to use the backprop on them only
so we use the loop for that indices to be targeted ,like looping on them
because we get the list of the indices and then with that indices we need to use the backprop individually right

there is a chance of the
accumulating so that we need to add them
self.weights.grad[idx]+=grad_out[i]
this works in the way we can say that
i -> indices index
idx - > that specific row
so we need to know the grad_out[index] 
how much does it need to change
and then we need to accumulate to that same row

for i,row_idx in enumerate(self.indices):
            self.weights.grad[row_idx]+=grad_out[i]
            
i->index this is what 0,1,2,3,4 
row_idx is what we go through the indices [0,2,3,5,0] so as the row_idx is 0 two times now the grad[row_idx] accumulates the gardient at the index 
for _ in indices (it will iterate over the values)
we accumulate to the same row,if there same row comes again ,and if not it will start adding fresh


if you wanted to use the torch code then
def backward(self, grad_out):
    # Initialize gradient tensor if None
    if self.weights.grad is None:
        self.weights.grad = torch.zeros_like(self.weights)
    
    # index_add_ - the PyTorch way!
    self.weights.grad.index_add_(0, self.indices, grad_out)
    
"""


import torch 

class TokenEmbeddings:
    
    def __init__(self,vocab_size,num_dims):
        self.weights = torch.randn(vocab_size,num_dims,requires_grad=True) 
        
        # Using nn.Parameter would register these weights with PyTorch so they appear in model.parameters() and get updated by optimizers.
        # Since this is a raw implementation, I'm using requires_grad=True manually. In the module version, I'll use nn.Parameter
    
        
    def forward(self,indices):
        self.indices = indices
        return self.weights[indices]

    
    def backward(self,grad_out):
        for i,row_idx in enumerate(self.indices):
            self.weights.grad[row_idx]+=grad_out[i]

        
    # optional if you inherit the torch module 
    
    def parameters(self):
        return [self.weights]


"""

Positional Embeddings is very important as we can't truly rely on the token embeddings
because depends on the position of the word,the meaning changes
in a sentence ,a single word can give different meanings depends on its position
subject,object...

Ex:
The dog bit the man
The man bit the dog  (😂️ just for example)

"""

class PositionalEmbeddings:

    def __init__(self,seq_len,num_dims):
        self.weights  = torch.randn(seq_len,num_dims,requires_grad=True)

    def forward(self,positions):
        self.positions = positions
        return self.weights[positions]

    def backward(self,grad_out):
        for i,pos in enumerate(self.positions):
            self.weights.grad[pos]+=grad_out[i]
        
    def parameters(self):
        return [self.weights]



B = 4  #batch size (how many it needs to process at a time in parallel)
seq_len=24  #this is the total how many words can enter once 
vocab_size=1000
num_dims=64

# create the objects for those repsective classes 

toe = TokenEmbeddings(vocab_size,num_dims)
poe = PositionalEmbeddings(seq_len,num_dims)

# randint give the integer values,ranging from the 0 to vocab_size-1 and then it expects the shape of the embeddings too

tok_embed = toe.forward(torch.randint(0,vocab_size,(B,seq_len)))


# so for the positions,we use the arange as it is same as the foor loop of how it assigns the i values 
# we get the 1d tensor of [0,1,....23]
# use the unsqueeze to add the dimensions so it has a shape of the (1,24)
# so expand works like the repetion of the same 1 as B times and -1 says to keep that dim 


positions = torch.arange(seq_len).unsqueeze(0).expand(B,-1) 


pos_embed = poe.forward(positions)

x=  tok_embed+pos_embed

x.shape



torch.Size([4, 24, 64])